## **Lab04-Fine-Tuning an LLM using LoRA/QLoRA**

## 01.install the library

In [10]:
!pip install -q transformers
!pip install -q datasets
!pip install -q peft
!pip install -q trl
!pip install -q accelerate
!pip install -q bitsandbytes

## 02.check GPU Availability

In [4]:

import torch

# Check whether a GPU is available
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    # Display the GPU name
    print("GPU name:", torch.cuda.get_device_name(0))

    # Calculate the GPU memory in GB
    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    gpu_memory = gpu_memory / (1024 ** 3)

    print("GPU memory:", round(gpu_memory, 2), "GB")


GPU available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


### What GPU is available?
- I have a Tesla T4 GPU with about 14.56 GB of GPU memory in my Google Colab environment.

### Why is a GPU useful for LLM fine-tuning?
- A GPU is good for LLM fine tuning because it can do lots of math operations in parallel. This helps with the speed of training the model as against using a CPU.

## 03.Select the language model

In [25]:
from transformers import BitsAndBytesConfig

# Set up 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [26]:
#Select and Load the Language Model
# Import the libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Select the pretrained instruction-tuned model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=quant_config
)

model.eval()
print("Model loaded successfully")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully


In [27]:

# Test the original model before fine-tuning
question = "How a university student can seek help with an assignment"
messages = [ {"role": "user", "content": question}]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert the prompt into tokens
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Generate an answer
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

# Decode only the generated answer
answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("Original model answer:", answer)


Question: How a university student can seek help with an assignment
Original model answer: Seeking help with an assignment is a common problem that many students face. Here are some steps you can take to get the support they need:

1. Identify the issue: Determine what specific aspect of your assignment or project you need help with. This could be anything from writing a research paper to completing a group project.

2. Research available resources: Look for resources and tools on how to find help with your assignment. This includes websites like Coursera, edX, and Khan Academy, as


### observation
succesfully Loaded a pretrained instruction-tuned language model. Before fine-tuning, I ran the original model on a question about university student support to see what it would say.

##04. Create the Dataset

In [9]:
# Create a question-and-answer dataset
# The topic is university student support

from datasets import Dataset

# Add 30 questions and answers about university student support
data = [
    {
        "question": "What is university student support?",
        "answer": "University student support is about helping students with academic, personal and university-related issues."
    },
    {
        "question": "How can a student get help with an assignment?",
        "answer": "A student can ask the lecturer for guidance, attend tutorials or contact the university academic support service."
    },
    {
        "question" : "What if a student misses a lecture?",
        "answer" : "The student should review the lecture content and reach out to the lecturer or another student to help catch them up."
    },
    {
        "question": "How can students manage their study time?",
        "answer": "Students can plan a study schedule, prioritise and break large tasks into smaller tasks."
    },
    {
        "question": "What is academic advising?",
        "answer": "Academic advising is advice on courses, academic progress and study-related decisions."
    },
    {
        "question": "How to contact a lecturer as a student?",
        "answer": "A student can contact a lecturer through university email, learning platform or during office hours."
    },
    {
        "question": "What should a student do if they are struggling with a subject?",
        "answer": "The student should go over the lecture notes, do practice questions and ask the lecturer or a tutor for help."
    },
    {
        "question": "How can students get ready for exams?",
        "answer": "Students should go over notes, practise past papers and follow a revision timetable."
    },
    {
        "question": "What is a university library used for?",
        "answer": "A university library provides books academic resources study spaces access to learning materials."
    },
    {
        "question": "Where can students find research articles?",
        "answer": "Students can access academic databases, Google Scholar and the university library's online resources."
    },
    {
       "question": "What is plagiarism?",
       "answer": "Plagiarism is the practice of using another person's work or ideas without giving credit to the original source."
    },
    {
        "question": "How can students avoid plagiarism?",
        "answer": "Students can avoid plagiarism by writing in their own words, citing sources and using the required referencing style."
    },
    {
        "question" : "What is group work?" ,
        "answer" : "Group work is an academic activity where students work together to complete a task or project."
    },
    {
        "question" : "How can students work effectively in a group?" ,
        "answer" : "Students can divide tasks fairly, communicate regularly and agree on deadlines."
    },
    {
         "question" : "What should a student do if a group member is not contributing?" ,
         "answer" : "The student should discuss the issue with the group and contact the lecturer if the problem continues."
    },
    {
        "question" : "How can students reduce exam stress?" ,
        "answer" : "Students can reduce exam stress by planning revision, taking breaks, sleeping well and asking for support."
    },
    {
        "question" : "Where can students seek emotional support?" ,
        "answer" : "Students can contact the university counselling service or speak with a trusted person."
    },
    {
        "question": "What is a student counselling service?",
        "answer": "A student counselling service provides support for personal, emotional and wellbeing concerns."
    },
    {
        "question": "How can students improve their attendance?",
        "answer": "Students can use reminders, plan their travel and organise their daily schedule."
    },
    {
        "question": "What should a student do if they are sick before an exam?",
        "answer": "The student should inform the university as soon as possible and follow its official medical or examination procedures."
    },
    {
        "question": "How can students access university announcements?",
        "answer": "Students can check the university learning platform, official email and notice boards."
    },
    {
        "question": "What is a student ID card used for?",
        "answer": "A student ID card may be used for identification and access to university facilities and services."
    },
    {
        "question": "How can students improve their presentation skills?",
        "answer": "Students can practise speaking, organise their content and prepare clear presentation slides."
    },
    {
        "question": "What should students do if they have a technical problem with the learning platform?",
        "answer": "Students should check their internet connection and contact the university IT support team if the problem continues."
    },
    {
        "question": "How can students improve their academic writing?",
        "answer": "Students can practise writing, organise their ideas clearly and use feedback to improve their work."
    },
    {
        "question": "What is a university career service?",
        "answer": "A university career service helps students with career planning, CV preparation and job or internship opportunities."
    },
    {
        "question": "How can students prepare for an internship?",
        "answer": "Students can prepare by updating their CV, practising interview questions and developing relevant skills."
    },
    {
        "question": "What should a student do if they cannot pay a university fee on time?",
        "answer": "The student should contact the university finance office and ask about available payment arrangements or support."
    },
    {
        "question": "How can students balance studies and personal life?",
        "answer": "Students can balance studies and personal life by planning their time, setting realistic goals and taking regular breaks."
    },
    {
        "question": "Why is asking for help important at university?",
        "answer": "Asking for help allows students to understand problems earlier and access the support they need to continue learning."
    }
]

# Convert the list into a Hugging Face dataset
dataset = Dataset.from_list(data)

# Check the number of examples
print("Number of question-answer pairs:", len(dataset))

# Display one example
print(dataset[0])

Number of question-answer pairs: 30
{'question': 'What is university student support?', 'answer': 'University student support is about helping students with academic, personal and university-related issues.'}


## 05. Configure LoRA using PEFT (QLoRA)

In [11]:
# Import the libraries
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

In [12]:
# Set up 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [13]:
# Load the model using 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [14]:
model = prepare_model_for_kbit_training(model)

In [15]:
# Configure LoRA parameters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

# Add LoRA adapters to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## 06.Fine-tune the model using SFTTrainer

In [18]:
# Format the question and answer
def format_chat(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]

    # Apply the chat template used by the model
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_dataset = dataset.map(format_chat)
# Display one formatted example
print(train_dataset[0]["text"])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is university student support?<|im_end|>
<|im_start|>assistant
University student support is about helping students with academic, personal and university-related issues.<|im_end|>



In [21]:
# Prepare the questions and answers for training

def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    return {"text": text}


dataset = dataset.map(format_example)

print(dataset.column_names)
print(dataset[0]["text"])

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

['question', 'answer', 'text']
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is university student support?<|im_end|>
<|im_start|>assistant
University student support is about helping students with academic, personal and university-related issues.<|im_end|>



In [33]:
# Import the training arguments
from transformers import TrainingArguments

# Set the training options
training_args = TrainingArguments(
    output_dir="./university_support_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    report_to="none"
)

In [31]:

# Prepare the model for LoRA training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

# Add LoRA adapters to the model
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [29]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [34]:
# Create the trainer
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

# Start fine-tuning
trainer.train()

Adding EOS to train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,3.734062
10,2.739265


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=12, training_loss=3.0938704013824463, metrics={'train_runtime': 19.0996, 'train_samples_per_second': 4.712, 'train_steps_per_second': 0.628, 'total_flos': 11686935169536.0, 'train_loss': 3.0938704013824463, 'entropy': 2.221641080720084, 'mean_token_accuracy': 0.6155053888048444, 'num_tokens': 5193.0, 'epoch': 3.0})

## 07.test the fine-tuned model

In [35]:
# Save the fine-tuned model
model.save_pretrained("./university_support_lora")
tokenizer.save_pretrained("./university_support_lora")

('./university_support_lora/tokenizer_config.json',
 './university_support_lora/chat_template.jinja',
 './university_support_lora/tokenizer.json')

In [36]:

# Test the fine-tuned model
question = "How can a university student seek help with an assignment?"

messages = [{"role": "user", "content": question}]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Fine-tuned model answer:")
print(answer)


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:242: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Fine-tuned model answer:
Aаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратаратарат


In [38]:
question = "What should a student do if they feel overwhelmed before exams?"
messages = [
    {"role": "user", "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate an answer
model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Get only the generated answer
answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("Fine-tuned model answer:")
print(answer)

Question: What should a student do if they feel overwhelmed before exams?
Fine-tuned model answer:
A student can take some steps to manage their stress and anxiety:

1. Set realistic goals: Break down the exam into smaller tasks that you can accomplish in one day.

2. Get enough sleep: Sleep is essential for your brain's function and helps you stay calm and focused.

3. Eat well: Eating a balanced diet with plenty of fruits and vegetables will help you maintain energy levels throughout the day.

4. Exercise regularly: Physical activity releases endorphins, which can help reduce stress and anxiety


## 08.Compare the original model

In [40]:
# Load the original model again
original_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

original_model.eval()

print("Original model loaded")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Original model loaded


In [45]:
# Test the original model with the same question

messages = [
    {"role": "user", "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = original_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("Original model answer:")
print(answer)

Question: How can students manage their time during a busy semester?
Original model answer:
Managing time effectively is crucial for students to ensure they have enough time to complete assignments, study, and maintain a healthy balance between academics and extracurricular activities. Here are some strategies that can help students manage their time efficiently:

1. **Set Clear Goals**: Define what you want to achieve in each class period or assignment. This helps in focusing on the most important tasks.

2. **Create a Schedule**: Use a planner or digital calendar to schedule your classes, exams, and other commitments.


In [46]:
question = "How can a student prepare for a group presentation?"

# Test the fine-tuned model

messages = [
    {"role": "user", "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("Fine-tuned model answer:")
print(answer)



Question: How can a student prepare for a group presentation?
Fine-tuned model answer:
A group presentation is an opportunity to showcase your skills and knowledge in front of the audience. Here are some tips on how to prepare for a group presentation:

1. Choose a topic: Select a topic that you are passionate about or have researched thoroughly. This will help you stay focused and engaged during the presentation.

2. Prepare your slides: Make sure your slides are well-designed and easy to read. Use bullet points to highlight key information and avoid cluttering them with too much text.

3. Practice


## Model Evaluation and Comparison

- The identical query was used to test the original model and the refined model in order to compare their answers.

- The original approach offered a general solution for time management during a demanding semester.

- Fine-Tuned Model: Based on the student-support data utilised during fine-tuning, the fine-tuned model produced an answer.

comparision :
Relevance, clarity, and the quality of the answers were used to compare the responses. This comparison made it easier to assess how fine-tuning affected the model's output.